# GeoSentinel-AI Final Phase (Phase 3 & 4)
This notebook trains the Baseline ImageNet Model (Phase 3) and evaluates both models (Phase 4).

### Pre-requisites:
1. **CRITICAL:** Ensure you have pushed your latest local code (with the deadlock fix) to GitHub!
2. Upload your `deeplabv3plus_best` and `change_unet_best` files as a single Kaggle Dataset and attach it to this notebook.
3. Turn on the **GPU T4 x2** accelerator.

In [ ]:
!pip install torch torchvision torchgeo lightning segmentation-models-pytorch rasterio pystac-client planetary-computer

In [ ]:
import os
import shutil

if os.path.exists('GeoSentinel-AI'):
    shutil.rmtree('GeoSentinel-AI')

!git clone https://github.com/karthikeya-bhamidipati/GeoSentinel-AI.git
os.chdir('GeoSentinel-AI')

In [ ]:
# 1. SMART COPY: Automatically find and copy the Elite weights from Kaggle input
import os
import shutil

target_weights_path = "data/weights"
os.makedirs(target_weights_path, exist_ok=True)

found_weights = 0
for root, _, files in os.walk("/kaggle/input"):
    for file in files:
        if file.endswith('.pt') or file.endswith('.ckpt'):
            source = os.path.join(root, file)
            destination = os.path.join(target_weights_path, file)
            shutil.copy(source, destination)
            print(f"Successfully copied {file} to {target_weights_path}")
            found_weights += 1

if found_weights >= 2:
    print("Elite weights staged successfully!")
else:
    print(f"ERROR: Found {found_weights} weights. Expected at least 2. Did you attach the dataset?")

In [ ]:
# 2. RUN PHASE 3: Train the Baseline Model
!python scripts/train_change.py --epochs 100 --batch-size 4 --num-workers 4 --ablation

In [ ]:
# 3. RUN PHASE 4: Evaluate Both Models
os.makedirs("outputs", exist_ok=True)
print("Evaluating Elite Model...")
!python scripts/evaluate_change.py --weights data/weights/change_unet_best.pt --batch-size 1 > outputs/elite_final_metrics.txt

print("Evaluating Baseline Model...")
!python scripts/evaluate_change.py --weights data/weights/change_unet_baseline_best.pt --ablation --batch-size 1 > outputs/baseline_final_metrics.txt

In [ ]:
# 4. Zip the final metrics for download
import os
from IPython.display import FileLink

os.chdir('/kaggle/working')
!zip -r Final_Metrics.zip GeoSentinel-AI/outputs/

print("\n--- DOWNLOAD FINAL METRICS ---")
display(FileLink(r'Final_Metrics.zip'))